# Phase 5: Full Observer Suite

Phase 5 is the final step in the GBD 2020 → GBD 2023 / vivarium 3 →
vivarium 4 build-up. It takes the complete Phase 4 stack (disease
models + risks + risk effects + healthcare / treatment / interventions)
and wires in the result-producing observers on top.

Model spec used:
`src/vivarium_nih_us_cvd/model_specifications/nih_us_cvd_phase5.yaml`.

New components exercised in this phase:

- vivarium_public_health 5 `MortalityObserver`, `DisabilityObserver`,
  and two `DiseaseObserver` instances (acute/chronic ischemic stroke
  and the IHD+HF disease model).
- The project's own `BinnedRiskObserver` for
  `risk_factor.high_systolic_blood_pressure` and
  `risk_factor.high_ldl_cholesterol`, which buckets
  exposure-time by clinical thresholds.
- The `SimpleResultsStratifier` / `ResultsStratifier` hierarchy,
  which patches the GBD 2023 age_bins table (drops the stray
  `index` column and redefines the youngest bin as `5_to_24`).

What this notebook verifies:

1. `InteractiveContext.setup()` succeeds — observers register
   adding-observations via the vivarium 4
   `register_adding_observation(..., requires_attributes=...)`
   API with `pop_filter='is_alive == True'`.
2. The stratifier produces the expected age bins for the GBD 2023
   artifact (no leftover `index` column, youngest bin merged into
   `5_to_24`, top bin extended so simulants cannot age past it
   within a single step).
3. Stepping the simulation accumulates observations every
   `time_step__prepare` / `collect_metrics` step. We run a short
   stretch of steps and then read the results via
   `sim.get_results()`.
4. The result tables have sensible shapes and totals — person-time
   adds up across the binned risk observers, deaths / ylls / ylds
   are non-negative, and disease transition counts show some motion.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vivarium import InteractiveContext

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

PHASE5_SPEC = '../src/vivarium_nih_us_cvd/model_specifications/nih_us_cvd_phase5.yaml'


## 1. Set up the simulation

If `InteractiveContext(...)` returns without raising, the Phase 5
observer machinery is wired up: each observer has registered its
adding-observations, the stratifier has registered the default
`age_group` / `sex` / `current_year` stratifications plus the
medication-adherence stratifications, and every `requires_attributes`
dependency has a resource producer in place.


In [2]:
sim = InteractiveContext(PHASE5_SPEC)
print('setup OK')
print('current time:', sim.current_time)
print('population size:', len(sim.get_population(['is_alive'])))


2026-04-11 15:32:36.799 | INFO     | simulation_1-artifact_manager:80 - Running simulation from artifact located at /home/abie/vivarium_nih_us_cvd/src/vivarium_nih_us_cvd/artifacts/united_states_of_america.hdf.


2026-04-11 15:32:36.800 | INFO     | simulation_1-artifact_manager:81 - Artifact base filter terms are [].


2026-04-11 15:32:36.800 | INFO     | simulation_1-artifact_manager:82 - Artifact additional filter terms are None.


2026-04-11 15:33:09.014 | WARNING  | simulation_1-lookup_table_manager:85 - Component 'risk_factor.outreach' configured, but didn't build lookup table 'exposure' during setup.


2026-04-11 15:33:09.015 | WARNING  | simulation_1-lookup_table_manager:85 - Component 'risk_factor.outreach' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.


...

2026-04-11 15:33:09.037 | WARNING  | simulation_1-lookup_table_manager:85 - Component 'risk_factor.ldlc_medication_adherence' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.


2026-04-11 15:33:09.038 | INFO     | simulation_1-results_context:131 - The following stratifications are registered but not used by any observers: 
['event_year']


2026-04-11 15:33:10.489 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.outreach.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.outreach.exposure_parameters.paf'.


2026-04-11 15:33:10.494 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.polypill.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.polypill.exposure_parameters.paf'.


setup OK
current time: 2024-01-01 00:00:00
population size: 1000


## 2. Stratifier sanity check

The `SimpleResultsStratifier.get_age_bins` override is what makes
the Phase 5 results machinery usable against the GBD 2023 USA
artifact. It:

1. Drops the stray `index` column that comes back from
   `reset_index()` on the artifact's MultiIndex age_bins table.
2. Keeps only bins with `age_start >= 25`, then prepends a
   synthetic `5_to_24` bin so we still have coverage for the
   youngest simulants.
3. Extends the top bin by one step so a simulant cannot age past
   it within a single 28-day step (MIC-4083 workaround).

We don't have the builder handy after setup, so we verify the
stratification indirectly after stepping (next section) by reading
back the distinct `age_group` values that appear in the result
tables.


In [3]:
age_range = sim.get_population(['age'])['age']
print('population adult age range:',
      round(age_range.min(), 2), '->', round(age_range.max(), 2))
print('age distribution summary:')
print(age_range.describe().round(2).to_string())


population adult age range: 5.02 -> 103.74
age distribution summary:
count    1000.00
mean       42.65
std        22.14
min         5.02
25%        23.59
50%        42.23
75%        61.09
max       103.74


## 3. Step the simulation and collect results

Run a handful of 28-day steps. Each step fires
`time_step__prepare` observations (the binned risk observers) and
then `collect_metrics` observations (the vph 5 disease / mortality
/ disability observers). After the run, `sim.get_results()` returns
a dict of result-name → DataFrame, one per registered
adding-observation.


In [4]:
N_STEPS = 6
for step_i in range(N_STEPS):
    sim.step()
print(f'{N_STEPS} steps OK, now at {sim.current_time.date()}')


2026-04-11 15:33:11.002 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2024-01-01 00:00:00


2026-04-11 15:33:11.371 | WARNING  | simulation_1-population_manager:747 - The 'affected_unmodeled.cause_specific_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'affected_unmodeled.cause_specific_mortality_rate.paf'.


2026-04-11 15:33:11.378 | WARNING  | simulation_1-population_manager:747 - The 'acute_ischemic_stroke.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_ischemic_stroke.excess_mortality_rate.paf'.


2026-04-11 15:33:11.391 | WARNING  | simulation_1-population_manager:747 - The 'chronic_ischemic_stroke.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'chronic_ischemic_stroke.excess_mortality_rate.paf'.


2026-04-11 15:33:11.398 | WARNING  | simulation_1-population_manager:747 - The 'acute_myocardial_infarction.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_myocardial_infarction.excess_mortality_rate.paf'.


...

2026-04-11 15:33:25.557 | WARNING  | simulation_1-population_manager:747 - The 'post_myocardial_infarction_to_acute_myocardial_infarction.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'post_myocardial_infarction_to_acute_myocardial_infarction.transition_rate.paf'.


2026-04-11 15:33:25.576 | WARNING  | simulation_1-population_manager:747 - The 'post_myocardial_infarction_to_heart_failure_from_ischemic_heart_disease.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'post_myocardial_infarction_to_heart_failure_from_ischemic_heart_disease.transition_rate.paf'.


2026-04-11 15:33:25.607 | WARNING  | simulation_1-population_manager:747 - The 'heart_failure_from_ischemic_heart_disease_to_acute_myocardial_infarction_and_heart_failure.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'heart_failure_from_ischemic_heart_disease_to_acute_myocardial_infarction_and_heart_failure.transition_rate.paf'.


2026-04-11 15:33:25.982 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.lifestyle.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.lifestyle.exposure_parameters.paf'.


6 steps OK, now at 2024-06-17


In [5]:
results = sim.get_results()
print(f'{len(results)} result tables registered')
print()
for name, df in results.items():
    print(f'  {name:70s} shape={df.shape}')

# Age-group stratification: pick any vph-observer result and read
# off the distinct age_group categories. We expect the synthetic
# `5_to_24` bin at the bottom and the standard GBD bins above.
sample = results['deaths']
print()
print('distinct age groups seen in deaths table:')
for ag in sorted(sample['age_group'].unique()):
    print(f'  {ag}')


15 result tables registered

  total_exposure_time_risk_high_ldl_cholesterol_below_2.59               shape=(64, 4)
  total_exposure_time_risk_high_ldl_cholesterol_between_2.59_and_3.36    shape=(64, 4)
  total_exposure_time_risk_high_ldl_cholesterol_between_3.36_and_4.14    shape=(64, 4)
  total_exposure_time_risk_high_ldl_cholesterol_between_4.14_and_4.91    shape=(64, 4)
  total_exposure_time_risk_high_ldl_cholesterol_above_4.91               shape=(64, 4)
  total_exposure_time_risk_high_systolic_blood_pressure_below_130.0      shape=(64, 4)
  total_exposure_time_risk_high_systolic_blood_pressure_between_130.0_and_140.0 shape=(64, 4)
  total_exposure_time_risk_high_systolic_blood_pressure_above_140.0      shape=(64, 4)
  deaths                                                                 shape=(512, 8)
  ylls                                                                   shape=(512, 8)
  ylds                                                                   shape=(512, 8)
  pe

## 4. Binned risk observer totals

The `BinnedRiskObserver` counts person-time (in years) inside each
exposure-threshold bin, stratified by age / sex / year. Summing
across a risk's bins should give us something close to
`population_size * simulated_time` (minus any time outside the
stratified universe).

For LDL-C we expect bins split at `2.59 / 3.36 / 4.14 / 4.91`
mmol/L. For SBP we expect bins split at `130 / 140` mmHg. The
totals should be on the order of 1000 simulants × ~6/13 of a
year ≈ 460 person-years.


In [6]:
def total_pt(prefix):
    return sum(
        float(df['value'].sum())
        for name, df in results.items()
        if name.startswith(prefix)
    )

ldlc_pt = total_pt('total_exposure_time_risk_high_ldl_cholesterol')
sbp_pt = total_pt('total_exposure_time_risk_high_systolic_blood_pressure')
print(f'LDL-C total person-time (years): {ldlc_pt:.2f}')
print(f'SBP   total person-time (years): {sbp_pt:.2f}')

print()
print('LDL-C bin split:')
for name, df in results.items():
    if name.startswith('total_exposure_time_risk_high_ldl_cholesterol'):
        print(f'  {name:80s} sum={df["value"].sum():.2f}')

print()
print('SBP bin split:')
for name, df in results.items():
    if name.startswith('total_exposure_time_risk_high_systolic_blood_pressure'):
        print(f'  {name:80s} sum={df["value"].sum():.2f}')


LDL-C total person-time (years): 458.89
SBP   total person-time (years): 458.89

LDL-C bin split:
  total_exposure_time_risk_high_ldl_cholesterol_below_2.59                         sum=245.00
  total_exposure_time_risk_high_ldl_cholesterol_between_2.59_and_3.36              sum=94.90
  total_exposure_time_risk_high_ldl_cholesterol_between_3.36_and_4.14              sum=79.80
  total_exposure_time_risk_high_ldl_cholesterol_between_4.14_and_4.91              sum=29.44
  total_exposure_time_risk_high_ldl_cholesterol_above_4.91                         sum=9.74

SBP bin split:
  total_exposure_time_risk_high_systolic_blood_pressure_below_130.0                sum=334.31
  total_exposure_time_risk_high_systolic_blood_pressure_between_130.0_and_140.0    sum=42.47
  total_exposure_time_risk_high_systolic_blood_pressure_above_140.0                sum=82.10


## 5. Mortality / disability totals

`MortalityObserver` and `DisabilityObserver` produce one row per
cause × stratum. `deaths` counts death events, `ylls` and `ylds`
accumulate years-of-life-lost / years-lived-with-disability. Over
the short simulated window these will be small but strictly
non-negative.


In [7]:
for key in ['deaths', 'ylls', 'ylds']:
    df = results[key]
    total = df['value'].sum()
    print(f'{key:8s} total={total:10.4f}  rows={len(df)}')
    nonzero = df[df['value'] > 0]
    if len(nonzero):
        top = (nonzero.groupby('sub_entity', observed=True)['value']
                        .sum().sort_values(ascending=False))
        print('  breakdown by cause:')
        for idx, val in top.head(5).items():
            print(f'    {idx:50s} {val:.4f}')
    else:
        print('  (no nonzero rows)')


deaths   total=    3.0000  rows=512
  breakdown by cause:
    heart_failure_from_ischemic_heart_disease          1.0000
    heart_failure_residual                             1.0000
    other_causes                                       1.0000
ylls     total=   73.2999  rows=512
  breakdown by cause:
    heart_failure_residual                             34.8795
    heart_failure_from_ischemic_heart_disease          20.4192
    other_causes                                       18.0012
ylds     total=    5.9094  rows=512
  breakdown by cause:
    all_causes                                         2.9547
    chronic_ischemic_stroke                            2.4061
    heart_failure_residual                             0.3939
    heart_failure_from_ischemic_heart_disease          0.1547
    acute_ischemic_stroke                              0.0001


## 6. Disease observer totals

`DiseaseObserver` produces two tables per disease model:
`person_time_<model>` (person-time in each disease state) and
`transition_count_<model>` (counts of transitions between states).
A healthy simulation will accumulate most of its person-time in
the susceptible state with some transitions into acute / chronic
states.


In [8]:
for disease in [
    'ischemic_stroke',
    'ischemic_heart_disease_and_heart_failure',
]:
    pt = results[f'person_time_{disease}']
    tc = results[f'transition_count_{disease}']
    print(f'== {disease} ==')
    print(f'  person-time total (years): {pt["value"].sum():.2f}  rows={len(pt)}')
    pt_by_state = (pt.groupby('sub_entity', observed=True)['value']
                      .sum().sort_values(ascending=False))
    print('  person-time by state:')
    for idx, val in pt_by_state.items():
        print(f'    {idx:60s} {val:.2f}')
    nonzero_tc = tc[tc['value'] > 0]
    if len(nonzero_tc):
        tc_by_trans = (nonzero_tc.groupby('sub_entity', observed=True)['value']
                                  .sum().sort_values(ascending=False))
        print('  observed transitions:')
        for idx, val in tc_by_trans.items():
            print(f'    {idx:60s} {val:.0f}')
    else:
        print('  (no transitions observed in this run)')
    print()


== ischemic_stroke ==
  person-time total (years): 459.12  rows=192
  person-time by state:
    susceptible_to_ischemic_stroke                               447.23
    chronic_ischemic_stroke                                      11.81
    acute_ischemic_stroke                                        0.08
  observed transitions:
    acute_ischemic_stroke_to_chronic_ischemic_stroke             1
    susceptible_to_ischemic_stroke_to_acute_ischemic_stroke      1

== ischemic_heart_disease_and_heart_failure ==
  person-time total (years): 459.12  rows=384
  person-time by state:
    susceptible_to_ischemic_heart_disease_and_heart_failure      453.06
    heart_failure_residual                                       2.99
    heart_failure_from_ischemic_heart_disease                    1.69
    post_myocardial_infarction                                   1.38
    acute_myocardial_infarction                                  0.00
    acute_myocardial_infarction_and_heart_failure                0.

## 7. Verdict

If this notebook ran end-to-end without exceptions and:

- The stratifier returns an `age_bins` table with a `5_to_24`
  bottom bin and no stray `index` column.
- All 15 result tables (`total_exposure_time_risk_high_ldl_cholesterol_*`
  × 5, `total_exposure_time_risk_high_systolic_blood_pressure_*`
  × 3, `deaths` / `ylls` / `ylds`, and
  `person_time_*` / `transition_count_*` for the two disease
  models) are populated with non-trivial values.
- The binned risk observer totals are within a factor of 2 of the
  expected `population_size × simulated_time` (here ~460 py).
- At least one transition is observed in each disease model over
  six 28-day steps.

then **Phase 5 is complete** and the incremental GBD 2023 /
vivarium 4 build-up plan has reached its final phase. The
remaining non-Phase work (wiring back `HealthcareVisitObserver`,
`CategoricalColumnObserver`, `LifestyleObserver`, and the
`JointPAFObserver` once `NonLogLinearMediatedRiskEffect` is in
place) is tracked separately.
